# Training: nested context-state encoder, centralized critic, and both actors (attack graph)
All components are trained jointly from scratch under centralized training with decentralized execution, alternating a representation step on Eq. (3) (`Train_Encoder`) with a game step on the critic and both actors (`Train_Critic`, `Train_Player1`, `Train_Player2`). During training, the defender's behavior policy is the critic's equilibrium row mixed with uniform exploration; the attacker plays its network and acts uniformly at random with probability $\lambda$, its irrationality level.

In [ ]:
%run Initialize.ipynb

## Settings

In [ ]:
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

GAMMA = 0.99
CRITIC_LR = 1e-5
PLAYER1_LR = 1e-4
PLAYER2_LR = 1e-5
ENCODER_LR = 5e-5
CRITIC_BATCH = 64
PLAYER1_BATCH = 128
PLAYER2_BATCH = 128
CONTEXT_BATCH = 256
TAU = 5e-4

# Eq. (3) weights and the Soft-InfoNCE temperature
LAMBDA_R = 0.25
LAMBDA_P = 0.25
LAMBDA_THETA = 0.25
SI_TEMPERATURE = 0.5

a_eps = 0.3
a_eps_decay = 0.99
a_eps_end = 0.1

num_episodes = 5000
episode_length = 50
Update_Every = 5

## Networks, optimizers, replay buffers

In [ ]:
# Representation: context encoder, its auxiliary heads, and EMA targets
Context_Model = TSTransformer().to(device)
trans_pred  = nn.Sequential(nn.Linear(Embedding_Size + n_a + d, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                            nn.Linear(256, Embedding_Size)).to(device)
reward_head = nn.Sequential(nn.Linear(Embedding_Size, 128), nn.ReLU(), nn.Linear(128, 1)).to(device)
proj_online = nn.Sequential(nn.Linear(Embedding_Size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
pred_online = nn.Sequential(nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
target_model = copy.deepcopy(Context_Model)
proj_target  = copy.deepcopy(proj_online)
for p_ in list(target_model.parameters()) + list(proj_target.parameters()):
    p_.requires_grad_(False)
target_model.eval(); proj_target.eval()
encoder_optimizer = optim.AdamW(list(Context_Model.parameters()) + list(trans_pred.parameters()) +
                                list(reward_head.parameters()) + list(proj_online.parameters()) +
                                list(pred_online.parameters()), lr=ENCODER_LR, amsgrad=True)

# Game: centralized critic and both actors
critic = Critic().to(device)
critic_target = Critic().to(device)
critic_target.load_state_dict(critic.state_dict())
critic_optimizer = optim.AdamW(critic.parameters(), lr=CRITIC_LR, amsgrad=True)

Player1 = DefenderActor().to(device)
player1_optimizer = optim.AdamW(Player1.parameters(), lr=PLAYER1_LR, amsgrad=True)
Player2 = AttackerActor().to(device)
player2_optimizer = optim.AdamW(Player2.parameters(), lr=PLAYER2_LR, amsgrad=True)

exp_memory = ReplayMemory(5000)
context_memory = ReplayMemory(500)
All_Loss = np.zeros([4, num_episodes])          # encoder, critic, defender, attacker
env_steps = 0

## Training

In [ ]:
T = lambda v, dtype=torch.float32: torch.tensor(v, dtype=dtype, device=device)

for i_episode in tqdm(range(num_episodes)):

    Theta, Type_Theta, D, rho = Sample_Context()
    x = np.random.randint(n)
    z_s = np.ones(d) / 2
    Context = np.zeros([Context_length, d * 3], dtype=np.float32)
    m2 = np.full(d, 0.2)
    counts = np.zeros(d)

    for t in range(episode_length):
        mask = Context_Mask(t)
        Context_Tensor = T(Context).view(1, Context_length, d * 3)
        Context_Model.eval()
        with torch.no_grad():
            Embeding = Context_Model(Context_Tensor, mask).squeeze(0)

        z_s_Tensor = T(z_s)
        x_Tensor = T(State_Matrix[x, :])
        theta_Tensor = T(Theta)
        type_theta_Tensor = T(Type_Theta)
        m2_Tensor = T(m2)

        with torch.no_grad():
            Q = critic(z_s_Tensor, Embeding, x_Tensor, m2_Tensor, type_theta_Tensor, theta_Tensor).cpu().numpy()
            _, a_dist, _ = MatrixValue(Q.reshape(n_a, n_u))
            a_dist = 0.8 * a_dist + 0.2 * np.full(n_a, 1 / n_a)
            u_dist = Player2(x_Tensor, m2_Tensor, type_theta_Tensor).cpu().numpy()
        a_dist = a_dist / a_dist.sum()
        u_dist = u_dist / u_dist.sum()

        if np.random.rand() < a_eps:
            a = int(random.choices(range(n_a), weights=np.ones([n_a]) / n_a)[0])
        else:
            a = int(np.random.choice(n_a, p=a_dist))

        if np.random.rand() < Type_Theta[2]:                  # irrational attacker acts uniformly with probability lambda
            u = int(random.choices(range(n_u), weights=np.ones(n_u) / n_u)[0])
        else:
            u = int(np.random.choice(n_u, p=u_dist))

        r = Reward(x, a, u, Theta, D, rho)
        next_x = Next_State(x, a, u, D, rho)
        next_m2, counts = m2_update(State_Matrix[x, :], State_Matrix[next_x, :], m2, counts)
        y = Observation(State_Matrix[next_x, :])
        next_z_s = Def_State_Blf(z_s, y, A[a])
        env_steps += 1

        new_row = Context_Row(z_s, a, next_z_s)
        if t < Context_length:
            Context[t, :] = new_row
        else:
            Context[:-1, :] = Context[1:, :]
            Context[-1, :]  = new_row

        if t > 0:
            next_mask = Context_Mask(t + 1)
            Next_Context_Tensor = T(Context).view(1, Context_length, d * 3)
            a_Tensor = T(a, torch.long)
            u_Tensor = T(u, torch.long)
            Reward_Tensor = T(r)
            exp_memory.push([x_Tensor.cpu(), T(State_Matrix[next_x, :]).cpu(), z_s_Tensor.cpu(), T(next_z_s).cpu(),
                             m2_Tensor.cpu(), T(next_m2).cpu(), a_Tensor.cpu(), u_Tensor.cpu(), Reward_Tensor.cpu(),
                             type_theta_Tensor.cpu(), theta_Tensor.cpu(),
                             Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu()])
            context_memory.push([Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu(),
                                 a_Tensor.cpu(), T(y, torch.long).cpu(), Reward_Tensor.cpu(), theta_Tensor.cpu()])

        x = next_x
        z_s = next_z_s
        m2 = next_m2

        if i_episode >= 20 and t % Update_Every == 0:
            All_Loss[0, i_episode] += Train_Encoder()          # representation step, Eq. (3)
            All_Loss[1, i_episode] += Train_Critic()           # game step
            All_Loss[2, i_episode] += Train_Player1()
            All_Loss[3, i_episode] += Train_Player2()

    a_eps = max(a_eps_end, a_eps * a_eps_decay)

## Save

In [ ]:
print(f"environment steps used in training: {env_steps:,d}")
torch.save(Context_Model.state_dict(), 'encoder.pth')
torch.save(critic.state_dict(), 'critic.pth')
torch.save(Player1.state_dict(), 'player1.pth')
torch.save(Player2.state_dict(), 'player2.pth')